# Source Validation — Stack Overflow Public Dataset

This notebook inspects the `bigquery-public-data.stackoverflow` dataset to:
1. Confirm the tables exist and match the expected schema
2. Sample rows from each source table
3. Validate key columns (nulls, grains, value distributions)
4. Estimate scan cost for the default year window

**Prerequisites:**
- `gcloud auth application-default login` completed
- `.env` file with `GCP_PROJECT_ID` set
- `pip install -r requirements.txt` in the project virtual environment

**This notebook is exploratory only.** No production logic lives here.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

# Load project .env (one level up from notebooks/)
env_path = Path("..") / ".env"
load_dotenv(dotenv_path=env_path)

GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]
STACKOVERFLOW_MIN_YEAR = int(os.getenv("STACKOVERFLOW_MIN_YEAR", "2021"))
SOURCE_PROJECT = "bigquery-public-data"
SOURCE_DATASET = "stackoverflow"

client = bigquery.Client(project=GCP_PROJECT_ID)

print(f"GCP project: {GCP_PROJECT_ID}")
print(f"Source: {SOURCE_PROJECT}.{SOURCE_DATASET}")
print(f"Year window: >= {STACKOVERFLOW_MIN_YEAR}")

## 1. List Available Tables in the Source Dataset

In [ ]:
tables = list(client.list_tables(f"{SOURCE_PROJECT}.{SOURCE_DATASET}"))
table_names = sorted([t.table_id for t in tables])

print(f"Tables in {SOURCE_PROJECT}.{SOURCE_DATASET} ({len(table_names)} total):")
for name in table_names:
    print(f"  - {name}")

## 2. Inspect Schema: posts_questions

In [ ]:
def get_schema_df(project, dataset, table):
    ref = client.get_table(f"{project}.{dataset}.{table}")
    rows = [
        {
            "column_name": field.name,
            "data_type": field.field_type,
            "mode": field.mode,
            "description": field.description or "",
        }
        for field in ref.schema
    ]
    return pd.DataFrame(rows)


schema_questions = get_schema_df(SOURCE_PROJECT, SOURCE_DATASET, "posts_questions")
print("posts_questions schema:")
schema_questions

## 3. Inspect Schema: posts_answers, users, tags

In [ ]:
for table_name in ["posts_answers", "users", "tags"]:
    schema = get_schema_df(SOURCE_PROJECT, SOURCE_DATASET, table_name)
    print(f"\n--- {table_name} ---")
    print(schema.to_string(index=False))

## 4. Sample Rows: posts_questions (year-window filtered)

In [ ]:
query_sample_questions = f"""
SELECT
    id,
    title,
    tags,
    score,
    view_count,
    answer_count,
    accepted_answer_id,
    owner_user_id,
    creation_date
FROM `{SOURCE_PROJECT}.{SOURCE_DATASET}.posts_questions`
WHERE EXTRACT(year FROM creation_date) >= {STACKOVERFLOW_MIN_YEAR}
LIMIT 5
"""

df_questions_sample = client.query(query_sample_questions).to_dataframe()
print(f"Sample questions (year >= {STACKOVERFLOW_MIN_YEAR}):")
df_questions_sample

## 5. Row Count Estimate by Year

In [ ]:
# NOTE: This query scans the full posts_questions table.
# Cost: ~60-70 GB. Run once to understand distribution; avoid repeating.
query_year_dist = f"""
SELECT
    EXTRACT(year FROM creation_date) AS year,
    COUNT(*) AS question_count
FROM `{SOURCE_PROJECT}.{SOURCE_DATASET}.posts_questions`
GROUP BY year
ORDER BY year DESC
"""

# Dry run to check bytes before executing
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run_job = client.query(query_year_dist, job_config=job_config)
bytes_gb = dry_run_job.total_bytes_processed / (1024**3)
cost_usd = (dry_run_job.total_bytes_processed / (1024**4)) * 6.25

print(f"Dry-run bytes: {bytes_gb:.1f} GB")
print(f"Estimated cost at $6.25/TB: ${cost_usd:.4f}")
print("\nUncomment the execute block below only if you accept this cost.")

# --- Uncomment to execute ---
# df_year_dist = client.query(query_year_dist).to_dataframe()
# df_year_dist

## 6. Validate Key Columns — Null Rates in Year Window

In [ ]:
query_null_check = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNTIF(id IS NULL) AS null_id,
    COUNTIF(title IS NULL) AS null_title,
    COUNTIF(tags IS NULL) AS null_tags,
    COUNTIF(score IS NULL) AS null_score,
    COUNTIF(view_count IS NULL) AS null_view_count,
    COUNTIF(answer_count IS NULL) AS null_answer_count,
    COUNTIF(owner_user_id IS NULL) AS null_owner_user_id,
    COUNTIF(accepted_answer_id IS NULL) AS null_accepted_answer_id,
    COUNTIF(creation_date IS NULL) AS null_creation_date
FROM `{SOURCE_PROJECT}.{SOURCE_DATASET}.posts_questions`
WHERE EXTRACT(year FROM creation_date) >= {STACKOVERFLOW_MIN_YEAR}
"""

df_null_check = client.query(query_null_check).to_dataframe()

# Transpose for readability
df_null_check_T = df_null_check.T.rename(columns={0: "count"})
total = df_null_check_T.loc["total_rows", "count"]
df_null_check_T["null_rate_pct"] = (df_null_check_T["count"] / total * 100).round(2)
df_null_check_T

## 7. Tag Format Validation

In [ ]:
query_tag_sample = f"""
SELECT
    id,
    tags,
    REPLACE(REPLACE(tags, '<', ''), '>', '|') AS tags_normalized,
    ARRAY_LENGTH(SPLIT(
        REGEXP_REPLACE(REPLACE(REPLACE(tags, '<', ''), '>', '|'), r'\\|$', ''),
        '|'
    )) AS tag_count
FROM `{SOURCE_PROJECT}.{SOURCE_DATASET}.posts_questions`
WHERE EXTRACT(year FROM creation_date) >= {STACKOVERFLOW_MIN_YEAR}
  AND tags IS NOT NULL
LIMIT 10
"""

df_tag_sample = client.query(query_tag_sample).to_dataframe()
df_tag_sample

## 8. User Table Validation

In [ ]:
query_user_check = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNTIF(id <= 0) AS system_stubs,
    COUNTIF(id > 0) AS real_users,
    COUNTIF(id > 0 AND reputation IS NULL) AS null_reputation,
    MIN(CASE WHEN id > 0 THEN reputation END) AS min_reputation,
    MAX(reputation) AS max_reputation,
    APPROX_QUANTILES(CASE WHEN id > 0 THEN reputation END, 4)[OFFSET(2)] AS median_reputation
FROM `{SOURCE_PROJECT}.{SOURCE_DATASET}.users`
"""

df_user_check = client.query(query_user_check).to_dataframe()
df_user_check.T.rename(columns={0: "value"})

## 9. Tags Table Validation

In [ ]:
query_tags_check = f"""
SELECT
    COUNT(*) AS total_tags,
    COUNTIF(tag_name IS NULL) AS null_tag_names,
    COUNTIF(count = 0) AS zero_count_tags,
    MAX(count) AS max_question_count,
    MIN(count) AS min_question_count
FROM `{SOURCE_PROJECT}.{SOURCE_DATASET}.tags`
"""

df_tags_check = client.query(query_tags_check).to_dataframe()
df_tags_check.T.rename(columns={0: "value"})

## 10. Top Tags in the Year Window

In [ ]:
query_top_tags = f"""
SELECT
    tag,
    COUNT(*) AS question_count
FROM `{SOURCE_PROJECT}.{SOURCE_DATASET}.posts_questions`,
UNNEST(SPLIT(
    REGEXP_REPLACE(REPLACE(REPLACE(tags, '<', ''), '>', '|'), r'\\|$', ''),
    '|'
)) AS tag
WHERE EXTRACT(year FROM creation_date) >= {STACKOVERFLOW_MIN_YEAR}
  AND tags IS NOT NULL
  AND TRIM(tag) != ''
GROUP BY tag
ORDER BY question_count DESC
LIMIT 20
"""

df_top_tags = client.query(query_top_tags).to_dataframe()
print(f"Top 20 tags (year >= {STACKOVERFLOW_MIN_YEAR}):")
df_top_tags

## 11. Accepted Answer Rate (Sample)

Quick check on the accepted-answer ratio for a recent year to validate
that `accepted_answer_id` is populated as expected.

In [ ]:
query_acceptance = f"""
SELECT
    EXTRACT(year FROM creation_date) AS year,
    COUNT(*) AS total_questions,
    COUNTIF(accepted_answer_id IS NOT NULL) AS questions_with_accepted_answer,
    ROUND(
        100.0 * COUNTIF(accepted_answer_id IS NOT NULL) / COUNT(*),
        2
    ) AS accepted_answer_rate_pct
FROM `{SOURCE_PROJECT}.{SOURCE_DATASET}.posts_questions`
WHERE EXTRACT(year FROM creation_date) >= {STACKOVERFLOW_MIN_YEAR}
GROUP BY year
ORDER BY year
"""

df_acceptance = client.query(query_acceptance).to_dataframe()
df_acceptance

## Summary

The cells above validate:
- Tables are accessible at the expected paths
- Schema matches the column list in `docs/source.md`
- Key columns have low null rates in the year window
- Tag normalization logic produces clean split-able strings
- Users table contains both real users and system stubs
- Tags table is small and suitable as a reference dimension

If these checks pass, the staging models in dbt should produce clean outputs.